# XAI (HPO-Tuned) -- Coffee Bean Quality Detection
### Verifikasi ulang "cara berpikir" 3 model pemenang `CBQD - HPO Optuna.ipynb`, sebelum peringkat barunya dipercaya

**Konteks:** `CBQD - HPO Optuna.ipynb` menghasilkan 3 checkpoint BARU (bobot berbeda dari
`CBQD - XAI.ipynb`) lewat retrain dengan hyperparameter hasil tuning Optuna:

| Model | Test macro-F1 (tuned) | vs baseline (pre-HPO) |
|---|---|---|
| `08_multitask` | 0.9698 | **+3,01pp** (naik signifikan) |
| `09_noise_robust` | 0.9610 | ~0 (tidak berubah) |
| `05_convnext_tiny` | 0.9483 | ~0 (tidak berubah) |

Checkpoint baru = optimization run baru = TIDAK otomatis mewarisi kesimpulan XAI yang
sudah diverifikasi di checkpoint lama (`CBQD - XAI.ipynb`) -- dua model dengan metrik akhir
mirip bisa saja mengandalkan fitur yang sama sekali berbeda (Rashomon effect). Lompatan
performa besar seperti `08_multitask` justru pola klasik shortcut learning pada dataset
kecil (~1.200 gambar) -- harus dicurigai, bukan langsung dirayakan.

**Strategi bertingkat (bukan re-run 72-cell penuh untuk 10 model seperti semula):**
- **`08_multitask` (prioritas tinggi, full battery):** Grad-CAM per head + head-agreement,
  causal probe (reposisi/occlusion/color-jitter), TCAV per head (Hipotesis #1/#4/#5),
  Hipotesis #2 (damage-leak), Hipotesis #3 (generalisasi `real_world/`). Delta yang besar
  butuh bukti paling lengkap.
- **`05_convnext_tiny` & `09_noise_robust` (spot-check, lebih murah):** metrik akhir nyaris
  tidak berubah dari baseline, jadi TIDAK diulang TCAV/causal-probe/embedding-lineage penuh.
  Cukup 3 bukti murah: (a) agreement rate prediksi checkpoint LAMA vs BARU di test set yang
  sama, (b) Hipotesis #2 dihitung ulang di checkpoint baru dibandingkan angka lama, (c)
  Hipotesis #3 dihitung ulang di checkpoint baru dibandingkan angka lama.

Model 01/02/03/04/06/07/10 TIDAK disentuh di notebook ini -- checkpoint & kesimpulan XAI-nya
tidak berubah karena HPO cuma menyentuh 3 model di atas.

Lihat juga `docs/xai-strategy.md` dan `CBQD - XAI.ipynb` (checkpoint pre-HPO, jadi baseline
pembanding notebook ini) serta `CBQD - HPO Optuna.ipynb` (sumber best hyperparameter).


## Section 1 -- Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)
# Catatan: hanya `captum` -- notebook ini tidak menyentuh Model 01/06/07/10 (tidak butuh
# lightgbm/shap/timm seperti CBQD - XAI.ipynb yang meng-cover 10 model).

!pip install -q captum

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")


In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest + checkpoint LAMA (pre-HPO, untuk baseline pembanding) dari R2

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

CKPT_DIR = Path("models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
existing_ckpts = list(CKPT_DIR.glob("*"))
print(f"Checkpoint yang sudah ada (dari dvc pull): {len(existing_ckpts)} file")
for p in existing_ckpts:
    print(" -", p.name)


## Section 2 -- Konfigurasi

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + turunan epoch/sampel agregat

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = False  # <-- dry-run (v1) sudah diverifikasi bersih di Kaggle (kernel v2, COMPLETE, tanpa error), full run.

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
    N_AGG_PER_CLASS = 3      # sampel agregat per kelas untuk probe/TCAV
    N_QUAL = 4
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45       # sama seperti retrain final CBQD - HPO Optuna.ipynb (EPOCHS_PHASE2_FINAL)
    EARLY_STOP_PATIENCE = 10
    N_AGG_PER_CLASS = 999999  # efektif: pakai semua test set (231 gambar)
    N_QUAL = 8

IMG_SIZE = 224
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | epochs phase1/phase2={EPOCHS_PHASE1}/{EPOCHS_PHASE2} | N_AGG_PER_CLASS={N_AGG_PER_CLASS}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )


In [ ]:
# Sub-Step 2.2
# Tujuan: Muat best hyperparameter hasil HPO (metadata/hpo_final_summary.csv) per model

import pandas as pd

hpo_summary = pd.read_csv("metadata/hpo_final_summary.csv").set_index("model")


def get_best_params(model_name, keys):
    row = hpo_summary.loc[model_name]
    params = {}
    for k in keys:
        v = row[f"param_{k}"]
        params[k] = int(v) if k in ("batch_size", "scheduler_patience") else float(v)
    return params


SHARED_KEYS = ["lr_phase1", "lr_phase2", "weight_decay", "batch_size", "scheduler_factor", "scheduler_patience"]
bp_convnext = get_best_params("05_convnext_tiny", SHARED_KEYS)
bp_multitask = get_best_params("08_multitask", SHARED_KEYS + ["type_loss_weight"])
bp_noise_robust = get_best_params("09_noise_robust", SHARED_KEYS + ["label_smoothing", "mislabel_weight", "mistake_threshold"])
for _name, _bp in [("05_convnext_tiny", bp_convnext), ("08_multitask", bp_multitask), ("09_noise_robust", bp_noise_robust)]:
    print(f"[{_name}] best params: {_bp}")


## Section 3 -- Data: Manifest, Dataset, Transform, DataLoader

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test/real_world DataFrame

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
real_world_df = manifest[manifest["split"] == "real_world"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}  real_world={len(real_world_df)}")


In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform; DataLoader eval bersama (test/real_world) + make_loaders() untuk fit/val per model

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
EVAL_BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    def __init__(self, df, root_dir, transform, weights=None, label_fn=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        label_fn = label_fn if label_fn is not None else (lambda l: LABEL_TO_IDX[l])
        self.labels = [label_fn(l) for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


def make_loaders(dataset_cls, fit_kwargs, val_kwargs, batch_size):
    fit_loader = DataLoader(dataset_cls(fit_df, PREP_DIR, train_transform, **fit_kwargs),
                             batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(dataset_cls(val_df, PREP_DIR, eval_transform, **val_kwargs),
                             batch_size=batch_size, shuffle=False, num_workers=2)
    return fit_loader, val_loader


test_loader = DataLoader(BeanDataset(test_df, PREP_DIR, eval_transform),
                          batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)
real_world_loader = DataLoader(
    BeanDataset(real_world_df, PREP_DIR, eval_transform, label_fn=lambda l: 0),
    batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2,
)  # label_fn dummy -- real_world tidak punya ground truth
print("DataLoaders siap.")


## Section 4 -- Fungsi Utilitas Model (identik `CBQD - HPO Optuna.ipynb`)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() & evaluate_combined() -- dibutuhkan training (early stopping) & sanity check

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device, combine_fn=None, class_names=None):
    class_names = class_names if class_names is not None else CLASS_NAMES
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = combine_fn(outputs) if combine_fn is not None else outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds if isinstance(preds, list) else preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}


In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() + train_one_model_hpo()/train_multitask_hpo() -- hyperparameter sebagai ARGUMEN
# (bukan konstanta global), supaya bisa dipakai ulang dengan best_params hasil HPO

import copy


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def _train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()


def train_one_model_hpo(model, head_module, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, criterion=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1, weight_decay=weight_decay
    )
    for epoch in range(EPOCHS_PHASE1):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


@torch.no_grad()
def _mt_val_f1(model, loader, device):
    model.eval()
    preds, labels_flat = [], []
    for images, damage_labels, type_labels in loader:
        images = images.to(device)
        out_damage, out_type = model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
        preds.extend(list(combined))
        dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
        true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
        labels_flat.extend(true_flat.tolist())
    return f1_score(labels_flat, preds, average="macro", zero_division=0)


def train_multitask_hpo(model, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, type_loss_weight=1.0):
    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + type_loss_weight * ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=lr_phase1, weight_decay=weight_decay,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


In [ ]:
# Sub-Step 4.3
# Tujuan: build_model() -- HANYA convnext_tiny & efficientnet_b0 (satu-satunya arsitektur yang
# dipakai 3 model ini; tidak perlu mobilenet/resnet/deit/timm seperti CBQD - XAI.ipynb)

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)


In [ ]:
# Sub-Step 4.4
# Tujuan: handcrafted_features() -- 11 fitur EDA, dipakai mistake_score (09_noise_robust) & concept split TCAV

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    else:
        area_frac = bbox_ratio = center_offset = np.nan
        bbox = (0, 0, w, h)

    edges = cv2.Canny(gray, 100, 200)
    feats = {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }
    return feats, bbox


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p)[0] for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


## Section 5 -- Retrain 3 Model dengan Best Hyperparameter HPO, Simpan Checkpoint TUNED

Checkpoint disimpan dengan akhiran `_tuned` (mis. `05_convnext_tiny_tuned.pt`) -- terpisah
dari checkpoint LAMA (`05_convnext_tiny.pt`, dari `CBQD - XAI.ipynb`, pre-HPO) yang masih
dipakai sebagai baseline pembanding di Section 8. Pola cek-checkpoint-dulu identik
`CBQD - XAI.ipynb` Section 5: kalau checkpoint tuned sudah ada (dari run sebelumnya), muat
langsung tanpa retrain.

In [ ]:
# Sub-Step 5.1
# Tujuan: get_or_train_tuned_cnn() -- retrain (atau muat) checkpoint TUNED untuk arsitektur CNN plain

retrained_any = False


def get_or_train_tuned_cnn(name, arch, bp, fit_loader_, val_loader_, criterion=None):
    global retrained_any
    path = CKPT_DIR / f"{name}_tuned.pt"
    model, head = build_model(arch, 4)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"[{name}_tuned] checkpoint dimuat dari {path}")
        return model.to(device).eval()
    model, val_f1 = train_one_model_hpo(
        model, head, fit_loader_, val_loader_, device, f"{name}_tuned",
        lr_phase1=bp["lr_phase1"], lr_phase2=bp["lr_phase2"], weight_decay=bp["weight_decay"],
        scheduler_factor=bp["scheduler_factor"], scheduler_patience=bp["scheduler_patience"],
        epochs_phase2=EPOCHS_PHASE2, criterion=criterion,
    )
    torch.save(model.state_dict(), path)
    retrained_any = True
    print(f"[{name}_tuned] selesai dilatih ulang, val_macro_f1={val_f1:.4f}")
    return model.eval()


fit_loader_cvt, val_loader_cvt = make_loaders(BeanDataset, {}, {}, bp_convnext["batch_size"])
model_convnext_tuned = get_or_train_tuned_cnn("05_convnext_tiny", "convnext_tiny", bp_convnext,
                                               fit_loader_cvt, val_loader_cvt)
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.2
# Tujuan: Precompute mistake_score (identik CBQD - HPO Optuna.ipynb) + retrain 09_noise_robust TUNED

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

X_fit_m9, y_fit_m9 = build_feature_matrix(fit_df)
_sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
_rf_noise = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
_proba_oof = cross_val_predict(_rf_noise, X_fit_m9, y_fit_m9, cv=_sgkf_noise,
                                groups=fit_df["cluster_id"].values, method="predict_proba")
_true_proba = _proba_oof[np.arange(len(y_fit_m9)), y_fit_m9]
_max_proba = _proba_oof.max(axis=1)
MISTAKE_SCORE = _max_proba - _true_proba

flagged_mask = MISTAKE_SCORE > bp_noise_robust["mistake_threshold"]
sample_weights_fit = np.where(flagged_mask, bp_noise_robust["mislabel_weight"], 1.0)
print(f"[09_noise_robust] kandidat mislabel (threshold={bp_noise_robust['mistake_threshold']:.4f}): "
      f"{flagged_mask.sum()} / {len(fit_df)}")

fit_loader_nr, val_loader_nr = make_loaders(BeanDataset, {"weights": sample_weights_fit.tolist()}, {},
                                             bp_noise_robust["batch_size"])
criterion_nr = nn.CrossEntropyLoss(label_smoothing=bp_noise_robust["label_smoothing"], reduction="none")
model_noise_robust_tuned = get_or_train_tuned_cnn("09_noise_robust", "efficientnet_b0", bp_noise_robust,
                                                   fit_loader_nr, val_loader_nr, criterion=criterion_nr)
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.3
# Tujuan: get_or_train_tuned_multitask() -- retrain (atau muat) 08_multitask TUNED

def get_or_train_tuned_multitask(bp):
    global retrained_any
    path = CKPT_DIR / "08_multitask_tuned.pt"
    model = EfficientNetMultiTask()
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"[08_multitask_tuned] checkpoint dimuat dari {path}")
        return model.to(device).eval()
    fit_loader_mt, val_loader_mt = make_loaders(MultiTaskDataset, {}, {}, bp["batch_size"])
    model, val_f1 = train_multitask_hpo(
        model, fit_loader_mt, val_loader_mt, device, "08_multitask_tuned",
        lr_phase1=bp["lr_phase1"], lr_phase2=bp["lr_phase2"], weight_decay=bp["weight_decay"],
        scheduler_factor=bp["scheduler_factor"], scheduler_patience=bp["scheduler_patience"],
        epochs_phase2=EPOCHS_PHASE2, type_loss_weight=bp["type_loss_weight"],
    )
    torch.save(model.state_dict(), path)
    retrained_any = True
    print(f"[08_multitask_tuned] selesai dilatih ulang, val_macro_f1={val_f1:.4f}")
    return model.eval()


model_multitask_tuned = get_or_train_tuned_multitask(bp_multitask)
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.4
# Tujuan: Push checkpoint tuned ke R2/DVC kalau ada yang baru dilatih

print(f"Ada model tuned yang baru dilatih ulang di sesi ini: {retrained_any}")
if retrained_any:
    os.system("dvc add models/checkpoints")
    os.system("dvc push")
    dvc_file = Path("models/checkpoints.dvc")
    if dvc_file.exists():
        print("--- Isi models/checkpoints.dvc (salin ke repo lokal, lalu commit) ---")
        print(dvc_file.read_text())
else:
    print("Semua checkpoint tuned sudah ada dari dvc pull -- tidak ada yang perlu di-push ulang.")


In [ ]:
# Sub-Step 5.5
# Tujuan: Sanity check -- verifikasi ulang test macro-F1 tuned model harus dekat dengan metadata/hpo_final_summary.csv
# (kalau meleset jauh, kemungkinan DRY_RUN masih True atau ada bug di retrain -- STOP sebelum lanjut ke XAI)

test_metrics_convnext = evaluate(model_convnext_tuned, test_loader, device)
print(f"[05_convnext_tiny tuned] test_macro_f1={test_metrics_convnext['macro_f1']:.4f}  "
      f"(hpo_final_summary.csv: {hpo_summary.loc['05_convnext_tiny', 'tuned_test_macro_f1']:.4f})")

test_metrics_noise_robust = evaluate(model_noise_robust_tuned, test_loader, device)
print(f"[09_noise_robust tuned] test_macro_f1={test_metrics_noise_robust['macro_f1']:.4f}  "
      f"(hpo_final_summary.csv: {hpo_summary.loc['09_noise_robust', 'tuned_test_macro_f1']:.4f})")


def multitask_predict_fn_eval(images):
    out_damage, out_type = model_multitask_tuned(images)
    damage_pred = out_damage.argmax(dim=1).cpu().numpy()
    type_pred = out_type.argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])


test_metrics_multitask = evaluate_combined(multitask_predict_fn_eval, test_loader, device)
print(f"[08_multitask tuned] test_macro_f1={test_metrics_multitask['macro_f1']:.4f}  "
      f"(hpo_final_summary.csv: {hpo_summary.loc['08_multitask', 'tuned_test_macro_f1']:.4f})")

if DRY_RUN:
    print("\nDRY_RUN=True -- angka di atas TIDAK diharapkan cocok (epoch sengaja dipangkas). "
          "Set DRY_RUN=False untuk verifikasi sebenarnya.")


## Section 6 -- Fungsi Utilitas XAI Bersama

Identik `CBQD - XAI.ipynb` Section 6, dipangkas ke yang benar-benar dipakai Section 7/8: TIDAK
ada embedding-lineage (NN label agreement) -- di notebook asli pun sub-model Family D (07/08)
tidak memakainya (backbone-nya sudah dicek di Family A), dan tier spot-check (05/09) sengaja
tidak mengulang seluruh baterai. TIDAK ada `xai_pipeline_neural()` generik -- Section 7/8
memanggil fungsi di bawah secara langsung sesuai kebutuhan masing-masing tier.

In [ ]:
# Sub-Step 6.1
# Tujuan: Grad-CAM (Captum) + Integrated Gradients + Occlusion -- dipakai Section 7 & 10

from captum.attr import LayerGradCam, LayerAttribution, IntegratedGradients, Occlusion


def gradcam_heatmaps(model, layer, images_tensor, target_classes):
    """images_tensor: (N,3,H,W) sudah di-normalize. Return list of (H,W) numpy heatmap."""
    lgc = LayerGradCam(model, layer)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = lgc.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]))
        a = LayerAttribution.interpolate(a, (IMG_SIZE, IMG_SIZE))
        heatmaps.append(a.squeeze().detach().cpu().numpy())
    return heatmaps


def ig_heatmaps(model, images_tensor, target_classes, n_steps=20):
    ig = IntegratedGradients(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = ig.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]), n_steps=n_steps)
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


def occlusion_heatmaps(model, images_tensor, target_classes, window=32, stride=16):
    occ = Occlusion(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = occ.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]),
                           sliding_window_shapes=(3, window, window), strides=(3, stride, stride))
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


In [ ]:
# Sub-Step 6.2
# Tujuan: TCAV manual: layer activation + CAV (logistic regression) + directional derivative -- dipakai Section 7.3 & damage_leak_test

from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_ind

GRAD_BATCH_SIZE = 32  # batch kecil supaya aman dari OOM (backward jauh lebih boros memori
                       # daripada forward-only), lalu hasil digabung.


def layer_activation_batch(model, layer, images_tensor):
    acts = {}
    def hook(m, i, o): acts["v"] = o.detach()
    h = layer.register_forward_hook(hook)
    outs = []
    with torch.no_grad():
        for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
            model(images_tensor[i:i + GRAD_BATCH_SIZE].to(device))
            a = acts["v"]
            a = a.mean(dim=[2, 3]) if a.dim() == 4 else (a.mean(dim=1) if a.dim() == 3 else a)
            outs.append(a.cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def layer_grad_wrt_target(model, layer, images_tensor, target_class):
    acts = {}
    def hook(m, i, o):
        o.retain_grad()
        acts["v"] = o
    h = layer.register_forward_hook(hook)
    outs = []
    for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
        batch = images_tensor[i:i + GRAD_BATCH_SIZE].to(device)
        out = model(batch)
        logit = out[:, target_class].sum()
        model.zero_grad(set_to_none=True)
        logit.backward()
        a = acts["v"]
        grad = a.grad
        grad = grad.mean(dim=[2, 3]) if grad.dim() == 4 else (grad.mean(dim=1) if grad.dim() == 3 else grad)
        outs.append(grad.detach().cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def compute_cav(pos_acts, neg_acts, seed=SEED):
    X = np.concatenate([pos_acts, neg_acts], axis=0)
    y = np.concatenate([np.ones(len(pos_acts)), np.zeros(len(neg_acts))])
    clf = LogisticRegression(max_iter=1000, random_state=seed).fit(X, y)
    cav = clf.coef_[0]
    return cav / (np.linalg.norm(cav) + 1e-8)


def tcav_score_with_significance(model, layer, concept_pos_imgs, concept_neg_imgs,
                                  target_imgs, target_class, n_random=5):
    pos_acts = layer_activation_batch(model, layer, concept_pos_imgs)
    neg_acts = layer_activation_batch(model, layer, concept_neg_imgs)
    cav = compute_cav(pos_acts, neg_acts)
    grads = layer_grad_wrt_target(model, layer, target_imgs, target_class)
    sensitivities = grads @ cav
    real_score = float((sensitivities > 0).mean())

    pool_acts = np.concatenate([pos_acts, neg_acts], axis=0)
    n_pos = len(pos_acts)
    rng = np.random.RandomState(SEED)
    random_scores = []
    for _ in range(n_random):
        perm = rng.permutation(len(pool_acts))
        rand_cav = compute_cav(pool_acts[perm[:n_pos]], pool_acts[perm[n_pos:]])
        rand_sens = grads @ rand_cav
        random_scores.append(float((rand_sens > 0).mean()))
    _, p_value = ttest_ind([real_score], random_scores) if len(set(random_scores)) > 1 else (np.nan, np.nan)
    return {"tcav_score": real_score, "random_scores_mean": float(np.mean(random_scores)),
            "random_scores": random_scores, "p_value": float(p_value) if p_value == p_value else None}


In [ ]:
# Sub-Step 6.3
# Tujuan: Probe kausal: reposisi bean, occlusion region, color jitter -- dipakai Section 7.2 (08_multitask)

def probe_reposition(orig_path, shift_frac=0.25, margin=0.2):
    _, bbox = handcrafted_features(orig_path)
    img = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    x0, y0, x1, y1 = bbox
    bw, bh = max(x1 - x0, 1), max(y1 - y0, 1)
    half = max(bw, bh) * (1 + margin) / 2

    def crop_at(cx, cy):
        xa, ya = int(max(cx - half, 0)), int(max(cy - half, 0))
        xb, yb = int(min(cx + half, w)), int(min(cy + half, h))
        if xb <= xa or yb <= ya:
            return Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE))
        return Image.fromarray(img[ya:yb, xa:xb]).resize((IMG_SIZE, IMG_SIZE))

    cx0, cy0 = (x0 + x1) / 2, (y0 + y1) / 2
    return crop_at(cx0, cy0), crop_at(cx0 + shift_frac * w, cy0)


def probe_occlude(pil_img, region="center"):
    arr = np.array(pil_img).copy()
    h, w = arr.shape[:2]
    fill = arr.reshape(-1, arr.shape[-1]).mean(axis=0).astype(arr.dtype)
    slices = {
        "center": (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4)),
        "left": (slice(0, h), slice(0, w // 2)),
        "right": (slice(0, h), slice(w // 2, w)),
    }[region]
    arr[slices] = fill
    return Image.fromarray(arr)


def probe_color_jitter(pil_img, brightness_factor=1.3, saturation_factor=1.3):
    """Perturbasi warna DI LUAR rentang augmentasi training (training cuma brightness
    +-10%, saturation +-5%) -- menguji apakah model rapuh terhadap perubahan warna
    yang lebih besar dari yang pernah dilihat saat training."""
    from PIL import ImageEnhance
    img = ImageEnhance.Brightness(pil_img).enhance(brightness_factor)
    img = ImageEnhance.Color(img).enhance(saturation_factor)
    return img


def prob_delta(predict_fn, pil_a, pil_b, class_idx):
    pa = predict_fn(pil_a)[class_idx]
    pb = predict_fn(pil_b)[class_idx]
    return float(abs(pa - pb)), float(pa), float(pb)


In [ ]:
# Sub-Step 6.4
# Tujuan: correlate_safe() (korelasi Pearson, aman dari NaN) + sample_per_class() -- dipakai Hipotesis #2 & sampling agregat

from scipy.stats import pearsonr


def correlate_safe(values, meta_values):
    values = np.asarray(values, dtype=float)
    meta_values = np.asarray(meta_values, dtype=float)
    valid = ~(np.isnan(values) | np.isnan(meta_values))
    if valid.sum() < 3 or np.std(values[valid]) < 1e-8 or np.std(meta_values[valid]) < 1e-8:
        return np.nan, np.nan
    r, p = pearsonr(values[valid], meta_values[valid])
    return float(r), float(p)


def sample_per_class(df, n_per_class, seed=SEED):
    parts = []
    for cls in CLASS_NAMES:
        sub_df = df[df["label"] == cls]
        n = min(n_per_class, len(sub_df))
        parts.append(sub_df.sample(n=n, random_state=seed))
    return pd.concat(parts).reset_index(drop=True)


In [ ]:
# Sub-Step 6.5
# Tujuan: Concept split untuk TCAV (Hipotesis #1/#4/#5) -- dihitung dari dataset_preprocessed
# (yang BENAR-BENAR dilihat model), dipakai TCAV per head 08_multitask (Section 7.3)

def compute_features_for_paths(paths):
    return pd.DataFrame([handcrafted_features(p)[0] for p in paths])


test_paths_prep = [PREP_DIR / p for p in test_df["image_path"]]
test_feats_prep = compute_features_for_paths(test_paths_prep)
test_feats_prep["image_path"] = test_df["image_path"].values
test_feats_prep["label"] = test_df["label"].values


def build_concept_examples(feature_col, concept_is_low, quantile=0.3, n=20):
    """concept_is_low=True -> positif = nilai fitur TERENDAH (mis. paling terpusat)."""
    vals = test_feats_prep[feature_col].values
    valid = ~np.isnan(vals)
    sub = test_feats_prep[valid].copy()
    lo_thr, hi_thr = sub[feature_col].quantile(quantile), sub[feature_col].quantile(1 - quantile)
    pos_df = sub[sub[feature_col] <= lo_thr] if concept_is_low else sub[sub[feature_col] >= hi_thr]
    neg_df = sub[sub[feature_col] >= hi_thr] if concept_is_low else sub[sub[feature_col] <= lo_thr]
    pos_df = pos_df.sample(n=min(n, len(pos_df)), random_state=SEED)
    neg_df = neg_df.sample(n=min(n, len(neg_df)), random_state=SEED)
    return pos_df["image_path"].tolist(), neg_df["image_path"].tolist()


CONCEPT_DEFS = [
    # Hipotesis #1: posisi bean sebagai shortcut framing (EDA v2 SS06)
    {"name": "off_center", "feature": "center_offset", "concept_is_low": True, "target_class": "premium"},
    {"name": "elongated_shape", "feature": "bbox_ratio", "concept_is_low": False, "target_class": "longberry"},
    # Hipotesis #4: warna sebagai artefak sesi pemotretan, bukan ciri bean asli --
    # mean_r/g/b adalah fitur dengan effect size TERBESAR di EDA (eta^2=0.22-0.24)
    {"name": "dark_color", "feature": "mean_r", "concept_is_low": True, "target_class": "defect"},
    # Hipotesis #5: ukuran-di-frame sebagai shortcut terpisah dari posisi/bentuk
    {"name": "large_area", "feature": "area_frac", "concept_is_low": False, "target_class": "peaberry"},
]
for cdef in CONCEPT_DEFS:
    pos_paths, neg_paths = build_concept_examples(cdef["feature"], cdef["concept_is_low"])
    cdef["pos_paths"] = pos_paths
    cdef["neg_paths"] = neg_paths
    print(f"Konsep '{cdef['name']}': {len(pos_paths)} positif, {len(neg_paths)} negatif")


In [ ]:
# Sub-Step 6.6
# Tujuan: predict_proba_pil() wrapper + FIG_DIR + heatmap overlay helper -- dipakai Section 7 & 10

def predict_proba_pil(model, pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        return torch.softmax(model(x), dim=1).cpu().numpy()[0]


import matplotlib
matplotlib.use("Agg")
import matplotlib.cm as cm
import matplotlib.pyplot as plt

FIG_DIR = Path("results/xai_hpo_tuned_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_VIZ_PER_CLASS = 1


def heatmap_overlay_rgb(pil_img, heatmap, alpha=0.5):
    img_arr = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")) / 255.0
    hm = heatmap - np.nanmin(heatmap)
    hm = hm / (np.nanmax(hm) + 1e-8)
    heat_colored = cm.jet(hm)[:, :, :3]
    overlay = (1 - alpha) * img_arr + alpha * heat_colored
    return np.clip(overlay, 0, 1)


In [ ]:
# Sub-Step 6.7
# Tujuan: load_pil_batch() + evaluasi real_world (confidence & distribusi kelas) -- dipakai Hipotesis #3

def load_pil_batch(paths, root_dir, transform):
    imgs = [transform(Image.open(root_dir / p if not str(p).startswith(str(root_dir)) else p).convert("RGB"))
            for p in paths]
    return torch.stack(imgs)


@torch.no_grad()
def predict_proba_loader(model, loader):
    model.eval()
    all_proba = []
    for images, _, _ in loader:
        all_proba.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(all_proba, axis=0)


def summarize_hypothesis3(name, test_proba, rw_proba):
    """test_proba/rw_proba selalu array (n, 4) probabilitas flat, apa pun mekanisme
    internal model penghasilnya (langsung softmax atau gabungan head damage x type)."""
    test_conf, rw_conf = test_proba.max(axis=1), rw_proba.max(axis=1)
    rw_pred = rw_proba.argmax(axis=1)
    rw_class_dist = {cname: float((rw_pred == i).mean()) for i, cname in enumerate(CLASS_NAMES)}
    print(f"--- {name} --- Confidence test: {test_conf.mean():.3f} | real_world: {rw_conf.mean():.3f}")
    print(f"  Distribusi prediksi real_world: {rw_class_dist}")
    return {"model": name, "test_confidence_mean": float(test_conf.mean()),
            "real_world_confidence_mean": float(rw_conf.mean()),
            "real_world_class_distribution": rw_class_dist}


def batch_predict_proba_pil_fn(predict_fn, df, root_dir):
    return np.array([
        predict_fn(Image.open(root_dir / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        for p in df["image_path"]
    ])


In [ ]:
# Sub-Step 6.8
# Tujuan: damage_leak_test() generik -- Hipotesis #2, dipakai model neural mana pun yang punya kelas defect

def damage_leak_test(model, layer, model_name, defect_idx):
    """CAV dilatih dari contoh defect (positif) vs non-defect (negatif) -- arah 'rusak' di
    ruang representasi model. Diuji: seberapa besar gambar NON-DEFECT sensitif ke arah itu,
    dan apakah sensitivitas tinggi berkorelasi dengan gambar itu benar-benar SALAH
    diklasifikasikan jadi defect."""
    defect_paths = test_df[test_df["label"] == "defect"]["image_path"].tolist()
    nondefect_df = test_df[test_df["label"] != "defect"]
    nondefect_paths_all = nondefect_df["image_path"].tolist()

    rng = np.random.RandomState(SEED)
    n_concept = min(30, len(defect_paths), len(nondefect_paths_all))
    defect_sample = rng.choice(defect_paths, n_concept, replace=False)
    nondefect_sample_for_cav = rng.choice(nondefect_paths_all, n_concept, replace=False)

    defect_imgs = load_pil_batch(list(defect_sample), PREP_DIR, eval_transform)
    nondefect_imgs_cav = load_pil_batch(list(nondefect_sample_for_cav), PREP_DIR, eval_transform)
    damage_cav = compute_cav(
        layer_activation_batch(model, layer, defect_imgs),
        layer_activation_batch(model, layer, nondefect_imgs_cav),
    )

    eval_df = sample_per_class(nondefect_df, N_AGG_PER_CLASS)
    eval_imgs = load_pil_batch(eval_df["image_path"].tolist(), PREP_DIR, eval_transform)
    grads = layer_grad_wrt_target(model, layer, eval_imgs, defect_idx)
    sensitivities = grads @ damage_cav

    predict_fn = lambda img: predict_proba_pil(model, img)
    predicted_idx = np.array([
        predict_fn(Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))).argmax()
        for p in eval_df["image_path"]
    ])
    misclassified_as_defect = (predicted_idx == defect_idx).astype(float)

    leak_score = float((sensitivities > 0).mean())
    r, p = correlate_safe(sensitivities, misclassified_as_defect)
    n_misclassified = int(misclassified_as_defect.sum())
    print(f"[{model_name}] Damage-leak score: {leak_score:.3f} | {n_misclassified}/{len(eval_df)} "
          f"non-defect justru diprediksi defect | r={r}, p={p}")
    return {"model": model_name, "damage_leak_score": leak_score,
            "n_misclassified_as_defect": n_misclassified, "n_eval": len(eval_df),
            "damage_leak_vs_misclass_r": r, "damage_leak_vs_misclass_p": p}


In [ ]:
# Sub-Step 6.9
# Tujuan: Muat hasil Hipotesis #2/#3 checkpoint LAMA (CBQD - XAI.ipynb, pre-HPO) sebagai baseline pembanding

old_hyp2 = pd.read_csv("metadata/xai_hypothesis2_damage_leak.csv").set_index("model")
old_hyp3 = pd.read_csv("metadata/xai_hypothesis3_real_world.csv").set_index("model")
print("Baseline lama (pre-HPO), damage_leak_score:")
print(old_hyp2.loc[["05_convnext_tiny", "09_noise_robust", "08_damage_head"], "damage_leak_score"])
print()
print("Baseline lama (pre-HPO), real_world_confidence_mean:")
print(old_hyp3.loc[["05_convnext_tiny", "09_noise_robust", "08_multitask"], "real_world_confidence_mean"])


## Section 7 -- Prioritas Tinggi: `08_multitask` (Tuned), Full Battery

Model ini naik +3,01pp macro-F1 setelah HPO -- lompatan terbesar dari 3 model yang di-tuning,
dan justru karena itu paling perlu diverifikasi. Baterai lengkap: Grad-CAM per head +
head-agreement, causal probe, TCAV per head, Hipotesis #2 (damage-leak), Hipotesis #3
(generalisasi `real_world/`) -- identik cakupan `CBQD - XAI.ipynb` Section 10 & 12 & 13 untuk
Model 08, dijalankan ulang di checkpoint TUNED dan dibandingkan ke angka checkpoint LAMA.

In [ ]:
# Sub-Step 7.1
# Tujuan: Grad-CAM per head (wrapper output tunggal) + head-agreement -- 08_multitask (tuned)

class HeadWrapper(nn.Module):
    def __init__(self, base, head_idx):
        super().__init__()
        self.base = base
        self.head_idx = head_idx

    def forward(self, x):
        outs = self.base(x)
        return outs[self.head_idx]


TYPE_LABEL_FN = lambda l: MultiTaskDataset.TYPE_MAP[l]

sample_nondefect = sample_per_class(test_df[test_df["label"] != "defect"], N_AGG_PER_CLASS)
images_nd = load_pil_batch(sample_nondefect["image_path"].tolist(), PREP_DIR, eval_transform)
type_targets = np.array([TYPE_LABEL_FN(l) for l in sample_nondefect["label"]])
damage_targets = np.zeros(len(sample_nondefect), dtype=int)  # semua non-defect -> target kelas "intact" (0)

damage_wrapper = HeadWrapper(model_multitask_tuned, 0).to(device).eval()
type_wrapper = HeadWrapper(model_multitask_tuned, 1).to(device).eval()
mt_layer = model_multitask_tuned.backbone.features[-1]

heatmaps_damage_08 = gradcam_heatmaps(damage_wrapper, mt_layer, images_nd, damage_targets)
heatmaps_type_08 = gradcam_heatmaps(type_wrapper, mt_layer, images_nd, type_targets)

head_corrs_08 = []
for ha, hb in zip(heatmaps_damage_08, heatmaps_type_08):
    if np.std(ha) < 1e-8 or np.std(hb) < 1e-8:
        continue
    head_corrs_08.append(float(np.corrcoef(ha.flatten(), hb.flatten())[0, 1]))
head_agreement_08 = float(np.mean(head_corrs_08)) if head_corrs_08 else np.nan
print(f"[08_multitask tuned] korelasi heatmap head-rusak vs head-tipe (BERBAGI 1 backbone) = "
      f"{head_agreement_08:.4f} ({'mirip -- wajar krn backbone dibagi' if head_agreement_08 > 0.5 else 'berbeda meski backbone sama'})")


In [ ]:
# Sub-Step 7.2
# Tujuan: Probe kausal 08_multitask (tuned) -- proba gabungan damage x type dari 1 backbone

def multitask_predict_proba_pil(pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out_damage, out_type = model_multitask_tuned(x)
        p_damage = torch.softmax(out_damage, dim=1)[0]
        p_type = torch.softmax(out_type, dim=1)[0]
    proba = np.zeros(4)
    proba[LABEL_TO_IDX["defect"]] = p_damage[1].item()
    not_defect = p_damage[0].item()
    proba[LABEL_TO_IDX["premium"]] = not_defect * p_type[0].item()
    proba[LABEL_TO_IDX["peaberry"]] = not_defect * p_type[1].item()
    proba[LABEL_TO_IDX["longberry"]] = not_defect * p_type[2].item()
    return proba


sample_08 = sample_per_class(test_df, N_AGG_PER_CLASS)
reposition_deltas_08, occlusion_deltas_08, color_deltas_08 = [], [], []
for _, row in sample_08.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    try:
        crop_a, crop_b = probe_reposition(RAW_DIR / row["orig_path"])
        d, _, _ = prob_delta(multitask_predict_proba_pil, crop_a, crop_b, cls_idx)
        reposition_deltas_08.append(d)
    except Exception:
        pass
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d2, _, _ = prob_delta(multitask_predict_proba_pil, prep_img, occluded, cls_idx)
    occlusion_deltas_08.append(d2)
    jittered = probe_color_jitter(prep_img)
    d3, _, _ = prob_delta(multitask_predict_proba_pil, prep_img, jittered, cls_idx)
    color_deltas_08.append(d3)

result_08 = {
    "model": "08_multitask_tuned",
    "head_agreement_heatmap_corr": head_agreement_08,
    "probe_reposition_mean_delta": float(np.mean(reposition_deltas_08)) if reposition_deltas_08 else np.nan,
    "probe_occlusion_center_mean_delta": float(np.mean(occlusion_deltas_08)),
    "probe_color_jitter_mean_delta": float(np.mean(color_deltas_08)),
}
print("08_multitask (tuned):", {k: v for k, v in result_08.items() if k != "model"})
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 7.3
# Tujuan: TCAV (Hipotesis #1/#4/#5) untuk head type & damage -- 08_multitask (tuned)

# type_wrapper cuma 3-kelas (premium/peaberry/longberry) -- dark_color->defect TIDAK ADA di
# ruang kelasnya (N/A arsitektural). damage_wrapper cuma 2-kelas (intact/defect) --
# off_center/elongated_shape/large_area TIDAK ADA di ruang kelasnya.
SUBMODEL_TCAV_TARGETS = {
    "type": {"off_center": 0, "elongated_shape": 2, "large_area": 1},
    "damage": {"dark_color": 1},
}


def run_tcav_on_submodel(model, layer, target_map, sample_labels_local, images_tensor):
    out = {}
    for cdef in CONCEPT_DEFS:
        local_idx = target_map.get(cdef["name"])
        if local_idx is None:
            out[cdef["name"]] = None
            continue
        pos_imgs = load_pil_batch(cdef["pos_paths"], PREP_DIR, eval_transform)
        neg_imgs = load_pil_batch(cdef["neg_paths"], PREP_DIR, eval_transform)
        target_mask = sample_labels_local == local_idx
        target_imgs = images_tensor[target_mask] if target_mask.sum() >= 2 else images_tensor
        out[cdef["name"]] = tcav_score_with_significance(model, layer, pos_imgs, neg_imgs, target_imgs, local_idx)
    return out


tcav_08_type = run_tcav_on_submodel(type_wrapper, mt_layer, SUBMODEL_TCAV_TARGETS["type"], type_targets, images_nd)
tcav_08_damage = run_tcav_on_submodel(damage_wrapper, mt_layer, SUBMODEL_TCAV_TARGETS["damage"], damage_targets, images_nd)
torch.cuda.empty_cache()

result_08["tcav_type_head"] = tcav_08_type
result_08["tcav_damage_head"] = tcav_08_damage

for label, res in [("type", tcav_08_type), ("damage", tcav_08_damage)]:
    summary = {k: (f"score={v['tcav_score']:.3f} p={v['p_value']}" if v else "N/A") for k, v in res.items()}
    print(f"[08_multitask tuned / head {label}] {summary}")


In [ ]:
# Sub-Step 7.4
# Tujuan: Hipotesis #2 (damage-leak) -- head-rusak 08_multitask (tuned), dibandingkan checkpoint lama

result_08_hyp2 = damage_leak_test(damage_wrapper, mt_layer, "08_damage_head_tuned", defect_idx=1)
old_score_hyp2_08 = float(old_hyp2.loc["08_damage_head", "damage_leak_score"])
result_08_hyp2["damage_leak_score_baseline"] = old_score_hyp2_08
result_08_hyp2["damage_leak_score_delta"] = result_08_hyp2["damage_leak_score"] - old_score_hyp2_08
print(f"[08_multitask] damage_leak_score: tuned={result_08_hyp2['damage_leak_score']:.4f} vs "
      f"baseline={old_score_hyp2_08:.4f} (delta={result_08_hyp2['damage_leak_score_delta']:+.4f})")
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 7.5
# Tujuan: Hipotesis #3 (generalisasi real_world) -- 08_multitask (tuned), dibandingkan checkpoint lama

test_proba_08 = batch_predict_proba_pil_fn(multitask_predict_proba_pil, test_df, PREP_DIR)
rw_proba_08 = batch_predict_proba_pil_fn(multitask_predict_proba_pil, real_world_df, PREP_DIR)
result_08_hyp3 = summarize_hypothesis3("08_multitask_tuned", test_proba_08, rw_proba_08)
old_conf_08 = float(old_hyp3.loc["08_multitask", "real_world_confidence_mean"])
result_08_hyp3["real_world_confidence_baseline"] = old_conf_08
result_08_hyp3["real_world_confidence_delta"] = result_08_hyp3["real_world_confidence_mean"] - old_conf_08
print(f"[08_multitask] real_world_confidence: tuned={result_08_hyp3['real_world_confidence_mean']:.4f} vs "
      f"baseline={old_conf_08:.4f} (delta={result_08_hyp3['real_world_confidence_delta']:+.4f})")


## Section 8 -- Spot-Check: `05_convnext_tiny` & `09_noise_robust` (Tuned vs Checkpoint Lama)

Delta test macro-F1 kedua model ini nyaris nol setelah HPO (-0,00002 dan +0,0004) -- kemungkinan
besar loss landscape-nya sudah dekat optimal sejak sebelum HPO. Tapi "kemungkinan besar" bukan
bukti, jadi tetap diverifikasi lewat 3 bukti murah (bukan re-run TCAV/causal-probe/embedding-
lineage penuh): (a) agreement rate prediksi checkpoint LAMA vs BARU pada test set yang sama,
(b) Hipotesis #2 dihitung ulang di checkpoint baru, (c) Hipotesis #3 dihitung ulang di checkpoint
baru -- keduanya dibandingkan ke angka checkpoint lama (`CBQD - XAI.ipynb`).

In [ ]:
# Sub-Step 8.1
# Tujuan: Bandingkan prediksi checkpoint LAMA (pre-HPO) vs BARU (tuned) di test set yang sama -- 05 & 09

def load_baseline_cnn(name, arch):
    path = CKPT_DIR / f"{name}.pt"
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint baseline {path} tidak ditemukan -- pastikan dvc pull sudah menariknya.")
    model, _ = build_model(arch, 4)
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()


baseline_convnext = load_baseline_cnn("05_convnext_tiny", "convnext_tiny")
baseline_noise_robust = load_baseline_cnn("09_noise_robust", "efficientnet_b0")


@torch.no_grad()
def prediction_agreement(model_old, model_new, loader):
    preds_old, preds_new, labels = [], [], []
    for images, y, _ in loader:
        images = images.to(device)
        preds_old.extend(model_old(images).argmax(1).cpu().tolist())
        preds_new.extend(model_new(images).argmax(1).cpu().tolist())
        labels.extend(y.tolist())
    preds_old, preds_new, labels = np.array(preds_old), np.array(preds_new), np.array(labels)
    return {
        "agreement_rate": float((preds_old == preds_new).mean()),
        "both_correct": float(((preds_old == labels) & (preds_new == labels)).mean()),
        "only_new_correct": float(((preds_new == labels) & (preds_old != labels)).mean()),
        "only_old_correct": float(((preds_old == labels) & (preds_new != labels)).mean()),
        "both_wrong": float(((preds_old != labels) & (preds_new != labels)).mean()),
    }


spotcheck_agreement = {
    "05_convnext_tiny": prediction_agreement(baseline_convnext, model_convnext_tuned, test_loader),
    "09_noise_robust": prediction_agreement(baseline_noise_robust, model_noise_robust_tuned, test_loader),
}
for name, res in spotcheck_agreement.items():
    print(f"[{name}] agreement={res['agreement_rate']:.4f}  both_correct={res['both_correct']:.4f}  "
          f"only_new_correct={res['only_new_correct']:.4f}  only_old_correct={res['only_old_correct']:.4f}  "
          f"both_wrong={res['both_wrong']:.4f}")


In [ ]:
# Sub-Step 8.2
# Tujuan: Hipotesis #2 (damage-leak) pada checkpoint TUNED 05 & 09 -- dibandingkan ke checkpoint lama

hypothesis2_tuned = []
for name, model in [("05_convnext_tiny", model_convnext_tuned), ("09_noise_robust", model_noise_robust_tuned)]:
    layer = model.features[-1]
    res = damage_leak_test(model, layer, f"{name}_tuned", LABEL_TO_IDX["defect"])
    old_score = float(old_hyp2.loc[name, "damage_leak_score"])
    res["damage_leak_score_baseline"] = old_score
    res["damage_leak_score_delta"] = res["damage_leak_score"] - old_score
    hypothesis2_tuned.append(res)
    torch.cuda.empty_cache()

hyp2_tuned_df = pd.DataFrame(hypothesis2_tuned)
print(hyp2_tuned_df.round(4).to_string(index=False))


In [ ]:
# Sub-Step 8.3
# Tujuan: Hipotesis #3 (generalisasi real_world) pada checkpoint TUNED 05 & 09 -- dibandingkan ke checkpoint lama

hypothesis3_tuned = []
for name, model in [("05_convnext_tiny", model_convnext_tuned), ("09_noise_robust", model_noise_robust_tuned)]:
    rw_proba = predict_proba_loader(model, real_world_loader)
    test_proba = predict_proba_loader(model, test_loader)
    res = summarize_hypothesis3(f"{name}_tuned", test_proba, rw_proba)
    old_conf = float(old_hyp3.loc[name, "real_world_confidence_mean"])
    res["real_world_confidence_baseline"] = old_conf
    res["real_world_confidence_delta"] = res["real_world_confidence_mean"] - old_conf
    hypothesis3_tuned.append(res)
    torch.cuda.empty_cache()

hyp3_tuned_df = pd.DataFrame(hypothesis3_tuned)
print(hyp3_tuned_df[["model", "test_confidence_mean", "real_world_confidence_mean",
                     "real_world_confidence_baseline", "real_world_confidence_delta"]].round(4).to_string(index=False))


## Section 9 -- Konsolidasi & Kesimpulan

In [ ]:
# Sub-Step 9.1
# Tujuan: Gabungkan seluruh temuan (08 full + 05/09 spot-check) jadi tabel ringkasan, simpan CSV

Path("metadata").mkdir(exist_ok=True)

hyp2_all = pd.concat([pd.DataFrame([result_08_hyp2]), hyp2_tuned_df], ignore_index=True)
hyp2_all.to_csv("metadata/xai_hpo_tuned_hypothesis2_damage_leak.csv", index=False)

hyp3_all = pd.concat([pd.DataFrame([result_08_hyp3]), hyp3_tuned_df], ignore_index=True)
hyp3_all.to_csv("metadata/xai_hpo_tuned_hypothesis3_real_world.csv", index=False)

summary_rows = [{
    "model": "08_multitask", "tier": "full (Grad-CAM+TCAV+probe+H2+H3)",
    "damage_leak_score": result_08_hyp2["damage_leak_score"],
    "damage_leak_delta_vs_baseline": result_08_hyp2["damage_leak_score_delta"],
    "real_world_confidence": result_08_hyp3["real_world_confidence_mean"],
    "real_world_confidence_delta_vs_baseline": result_08_hyp3["real_world_confidence_delta"],
    "old_vs_new_agreement_rate": None,
    "head_agreement_heatmap_corr": head_agreement_08,
}]
for row2, row3 in zip(hyp2_tuned_df.to_dict("records"), hyp3_tuned_df.to_dict("records")):
    name = row2["model"].replace("_tuned", "")
    summary_rows.append({
        "model": name, "tier": "spot-check (agreement+H2+H3)",
        "damage_leak_score": row2["damage_leak_score"],
        "damage_leak_delta_vs_baseline": row2["damage_leak_score_delta"],
        "real_world_confidence": row3["real_world_confidence_mean"],
        "real_world_confidence_delta_vs_baseline": row3["real_world_confidence_delta"],
        "old_vs_new_agreement_rate": spotcheck_agreement[name]["agreement_rate"],
        "head_agreement_heatmap_corr": None,
    })

xai_hpo_tuned_summary_df = pd.DataFrame(summary_rows)
xai_hpo_tuned_summary_df.to_csv("metadata/xai_hpo_tuned_summary.csv", index=False)
status_text = "BELUM final (sampel/epoch kecil)" if DRY_RUN else "hasil run penuh"
print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {status_text}")
print()
pd.set_option("display.max_columns", None, "display.width", 200)
print(xai_hpo_tuned_summary_df.round(4).to_string(index=False))


In [ ]:
# Sub-Step 9.2
# Tujuan: Interpretasi ringkas -- dibaca manual bersama tabel di atas

print("""
Cara membaca metadata/xai_hpo_tuned_summary.csv:
- damage_leak_delta_vs_baseline (Hipotesis #2) POSITIF & besar berarti checkpoint TUNED
  justru LEBIH mengandalkan arah representasi 'rusak' untuk gambar non-defect dibanding
  checkpoint lama -- ini sinyal kuat bahwa kenaikan recall/F1 setelah HPO didorong shortcut,
  bukan sinyal genuin. Delta di sekitar 0 (naik-turun kecil) berarti tidak ada bukti
  peningkatan ketergantungan pada shortcut ini.
- real_world_confidence_delta_vs_baseline (Hipotesis #3) NEGATIF & besar berarti checkpoint
  TUNED kurang generalisasi ke kondisi pengambilan gambar di luar test/ dibanding checkpoint
  lama -- indikasi overfit ke idiosinkrasi test set, bukan perbaikan genuin. Delta di sekitar
  0 atau positif berarti generalisasi ke real_world/ tidak memburuk (atau membaik).
- old_vs_new_agreement_rate (khusus tier spot-check 05/09) tinggi (>0,95) berarti checkpoint
  baru berperilaku hampir identik checkpoint lama pada level PREDIKSI PER-GAMBAR, bukan cuma
  metrik agregat -- bukti kuat bahwa temuan XAI CBQD - XAI.ipynb (TCAV/causal-probe/embedding-
  lineage) masih relevan untuk checkpoint tuned ini tanpa perlu diulang penuh.
- head_agreement_heatmap_corr (khusus 08_multitask) tinggi WAJAR karena head damage & type
  berbagi 1 backbone -- bukan sinyal shortcut, cuma konsekuensi arsitektural multi-task.
- tcav_*_score/p pada result_08 (lihat Sub-Step 7.3 print) mengikuti interpretasi yang sama
  seperti CBQD - XAI.ipynb Section 14.2: skor tinggi & p signifikan pada off_center/dark_color/
  elongated_shape/large_area berarti model 08_multitask (tuned) memakai fitur itu sebagai
  sinyal -- bandingkan ke xai_summary.csv (checkpoint lama) untuk melihat apakah tuning
  MENGUBAH konsep mana yang jadi andalan model, bukan cuma seberapa kuat.

Kesimpulan tentang pertanyaan "apakah model tetap tidak melakukan shortcut": lihat tanda delta
di atas, BUKAN angka absolutnya -- checkpoint tuned dipercaya sejauh delta-nya terhadap
checkpoint lama kecil/tidak mencurigakan, bukan sekadar karena test macro-F1-nya membaik.
""")


## Section 10 -- Visualisasi untuk Laporan

In [ ]:
# Sub-Step 10.1
# Tujuan: Grad-CAM + IG + Occlusion overlay -- 05_convnext_tiny & 09_noise_robust (tuned)

TUNED_MODELS_FAMILY_A = {
    "05_convnext_tiny": model_convnext_tuned,
    "09_noise_robust": model_noise_robust_tuned,
}
TUNED_LAYER = {name: model.features[-1] for name, model in TUNED_MODELS_FAMILY_A.items()}

viz_df = sample_per_class(test_df, N_VIZ_PER_CLASS)
viz_imgs = load_pil_batch(viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
viz_labels = np.array([LABEL_TO_IDX[l] for l in viz_df["label"]])
viz_pil_originals = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in viz_df["image_path"]]

for model_name, model in TUNED_MODELS_FAMILY_A.items():
    layer = TUNED_LAYER[model_name]
    gc_maps = gradcam_heatmaps(model, layer, viz_imgs, viz_labels)
    ig_maps_v = ig_heatmaps(model, viz_imgs, viz_labels)
    occ_maps_v = occlusion_heatmaps(model, viz_imgs, viz_labels)

    n = len(viz_df)
    fig, axes = plt.subplots(n, 4, figsize=(10, 2.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        axes[i, 0].imshow(viz_pil_originals[i]); axes[i, 0].set_ylabel(viz_df.iloc[i]["label"], fontsize=10)
        axes[i, 1].imshow(heatmap_overlay_rgb(viz_pil_originals[i], gc_maps[i]))
        axes[i, 2].imshow(heatmap_overlay_rgb(viz_pil_originals[i], ig_maps_v[i]))
        axes[i, 3].imshow(heatmap_overlay_rgb(viz_pil_originals[i], occ_maps_v[i]))
        for j in range(4):
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if i == 0:
            for j, t in enumerate(["Original", "Grad-CAM", "Integrated Gradients", "Occlusion"]):
                axes[i, j].set_title(t, fontsize=10)
    fig.suptitle(f"{model_name} (tuned)", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"gradcam_{model_name}_tuned.png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"Disimpan: {FIG_DIR / f'gradcam_{model_name}_tuned.png'}")


In [ ]:
# Sub-Step 10.2
# Tujuan: Visual Grad-CAM pada contoh real_world/ -- 09_noise_robust (tuned): tetap fokus ke bean?

rw_viz_df = real_world_df.sample(n=min(8, len(real_world_df)), random_state=SEED).reset_index(drop=True)
rw_viz_imgs = load_pil_batch(rw_viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
rw_viz_pil = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in rw_viz_df["image_path"]]

_rw_model = model_noise_robust_tuned
_rw_layer = TUNED_LAYER["09_noise_robust"]
rw_proba_viz = np.array([predict_proba_pil(_rw_model, img) for img in rw_viz_pil])
rw_pred_viz = rw_proba_viz.argmax(axis=1)
rw_heatmaps = gradcam_heatmaps(_rw_model, _rw_layer, rw_viz_imgs, rw_pred_viz)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for i in range(len(rw_viz_pil)):
    r, c = divmod(i, 4)
    axes[r, c].imshow(heatmap_overlay_rgb(rw_viz_pil[i], rw_heatmaps[i]))
    axes[r, c].set_title(f"pred={CLASS_NAMES[rw_pred_viz[i]]}\nP={rw_proba_viz[i][rw_pred_viz[i]]:.2f}", fontsize=9)
    axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
fig.suptitle("Grad-CAM pada real_world/ (09_noise_robust tuned) -- sanity check generalisasi", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "real_world_gradcam_09_noise_robust_tuned.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'real_world_gradcam_09_noise_robust_tuned.png'}")
